# Synthetic Oracle vs. Broken representation ablation (TSC scale)

This notebook implements the controlled synthetic experiment described in the paper's appendix
(**"Synthetic Oracle vs. Weak-Feature Ablation at TSC Scale"**).

Core idea:
- We generate a synthetic benchmark whose model-by-dataset performance is induced by a low-dimensional latent geometry.
- We then compare subset-selection methods in two representation regimes:
  1. **Oracle / aligned representation** — feature geometry closely matches the latent rank-inducing geometry.
  2. **Broken / weakly aligned representation** — only a small fraction of the feature signal is aligned with the true geometry.

Interpretation in the paper:
- If the dataset representation is genuinely aligned with the latent task geometry, geometry-based selection should preserve
  model rankings much better than Random.
- If the representation is mostly noise or only weakly aligned, the advantage of geometric selection should shrink or disappear.

Important notes about this notebook snapshot:
- The **final cell currently runs only one regime at a time**, controlled by `USE_MISALIGNED`.
  - `False` → Oracle / aligned representation
  - `True`  → Broken / weakly aligned representation
- The **outer loop currently iterates over 100 seeds**.
  The paper text describes aggregation over a larger number of independent seeds, so increase that loop if you want to match
  the final paper configuration exactly.
- The notebook keeps helper wrappers for A-/D-optimality, but those methods are **commented out in the synthetic ablation**
  and are not part of the final reported Oracle-vs-Broken comparison in the manuscript.


In [16]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import umap
import hdbscan
import matplotlib.pyplot as plt
import pickle
from tqdm.notebook import tqdm

In [ ]:
cur_dir = os.getcwd().split('\\')

if cur_dir[-1] == 'notebooks':
    os.chdir("..")

from utils.data_load_utilities.data_loader import load_model_results
from utils.get_global_const import get_global_const
from utils.get_metrics import get_metrics
from utils.get_ranks import get_ranks_s, get_ranks
from methods.ADoE_method import *
from methods.k_nearest_methods import *
from methods.kmeans_methods import *
from methods.opt_methods import *
from methods.sparce_methods import *
from methods.entrophy_metods import *

from testing_pipeline.testing_pipeline_stats import *

from sklearn.linear_model import LassoCV
from sklearn.feature_selection import mutual_info_regression

from datetime import datetime
import re

import warnings
warnings.simplefilter('ignore')

In [18]:
scores, datasets, models = get_global_const()
scores

Parsing Bakeoff2017 models...

Parsing Bakeoff2021 models...

Parsing Bakeoff2023 models...

Parsing HIVE-COTEV2 models...



{'1NN-DTW':                   folds:         0         1         2         3         4  \
 0                  Adiac  0.603581  0.603581  0.601023  0.606138  0.608696   
 1              ArrowHead  0.702857  0.742857  0.680000  0.708571  0.760000   
 2                   Beef  0.633333  0.533333  0.533333  0.500000  0.566667   
 3              BeetleFly  0.700000  0.800000  0.650000  0.800000  0.650000   
 4            BirdChicken  0.750000  0.850000  0.950000  0.800000  0.950000   
 ..                   ...       ...       ...       ...       ...       ...   
 107  EOGHorizontalSignal  0.441989  0.607735  0.566298  0.569061  0.624309   
 108    EOGVerticalSignal  0.433702  0.511050  0.549724  0.566298  0.563536   
 109                 Rock  0.540000  0.680000  0.660000  0.640000  0.720000   
 110                 Crop  0.710536  0.713631  0.706964  0.708929  0.712321   
 111            Chinatown  0.973761  0.909621  0.953353  0.944606  0.970845   
 
             5         6         7     

In [19]:
tsfresh_features = pd.read_csv(Path('data/datasets_features/tsfresh_important_features.csv'), index_col=0)
# tsfresh_features

In [20]:
# chosen_datasets = tsfresh_features.Name.values
chosen_datasets = sorted(datasets)
chosen_datasets[:5]

['ACSF1', 'Adiac', 'ArrowHead', 'BME', 'Beef']

In [21]:
features = pd.read_csv(Path('data/datasets_features/features.csv'), index_col=0)
features = features.set_index('Name')
features = features.loc[chosen_datasets, :]
features = features.reset_index()
features

,Name,entropy,gini,number_of_classes,size
0,ACSF1,3.321928,0.900000,1.0,200.0
1,Adiac,5.202000,0.972675,1.0,781.0
2,ArrowHead,1.576854,0.662833,1.0,211.0
3,BME,1.584963,0.666667,1.0,180.0
4,Beef,2.321928,0.800000,1.0,60.0
...,...,...,...,...,...
107,Wine,0.999473,0.499635,0.0,111.0
108,WordSynonyms,4.071877,0.909999,1.0,905.0
109,Worms,2.117034,0.734211,1.0,258.0
110,WormsTwoClass,0.982591,0.487981,0.0,258.0


In [22]:
sf = features.drop(columns=['Name']).copy()

In [23]:
scores_aggr = {
    model: model_score[model_score["folds:"].isin(chosen_datasets)].reset_index(drop=True) for model, model_score in scores.items()
}

scores_aggr = {
    model: model_score.set_index("folds:").loc[chosen_datasets, :].reset_index() for model, model_score in scores_aggr.items()
}

scores_aggr = {
    model: model_score[model_score.columns[1:]].mean(axis=1) for model, model_score in scores_aggr.items()
}

scores_aggr = pd.DataFrame(scores_aggr)

scores_aggr

,1NN-DTW,Arsenal,BOSS,CIF,CNN,Catch22,DrCIF,EE,FreshPRINCE,HC1,...,STSF,ShapeDTW,Signatures,TDE,TS-CHIEF,TSF,TSFresh,WEASEL-D,WEASEL,cBOSS
0,0.553667,0.805667,0.752667,0.766667,0.345333,0.795667,0.782667,0.571667,0.800000,0.848000,...,0.773333,0.486000,0.726333,0.804000,0.807000,0.667000,0.790000,0.829000,0.775667,0.775000
1,0.603410,0.769224,0.745695,0.766922,0.387383,0.697187,0.811679,0.657374,0.808355,0.790878,...,0.813384,0.653367,0.700256,0.754902,0.779710,0.715686,0.789770,0.817818,0.793606,0.745780
2,0.722095,0.864762,0.860000,0.823238,0.759810,0.760000,0.828000,0.860381,0.788381,0.877524,...,0.789143,0.856000,0.773333,0.897524,0.881143,0.784190,0.613143,0.885143,0.879429,0.881714
3,0.880889,0.997333,0.852889,0.997333,0.977778,0.895111,0.998889,0.975111,0.999111,0.970222,...,0.999556,0.874222,0.993778,0.920444,0.996444,0.960444,0.997111,0.992444,0.936222,0.769556
4,0.504444,0.751111,0.630000,0.792222,0.667778,0.484444,0.794444,0.515556,0.791111,0.718889,...,0.790000,0.530000,0.632222,0.664444,0.632222,0.677778,0.734444,0.776667,0.718889,0.577778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,0.859259,0.916049,0.891975,0.881481,0.537654,0.708642,0.884568,0.867284,0.896914,0.898148,...,0.880864,0.867901,0.868519,0.888272,0.898148,0.857407,0.497531,0.957407,0.901852,0.869136
108,0.646290,0.756688,0.649530,0.703762,0.561651,0.547597,0.705486,0.775653,0.629101,0.698380,...,0.663845,0.664107,0.598171,0.752978,0.793678,0.643887,0.590961,0.750784,0.672466,0.664002
109,0.574026,0.723377,0.726407,0.696104,0.451082,0.723377,0.749784,0.633333,0.772727,0.735931,...,0.713853,0.493506,0.639394,0.744156,0.768398,0.620346,0.747186,0.717749,0.794372,0.716450
110,0.675325,0.796104,0.812987,0.806926,0.605628,0.790909,0.807792,0.712987,0.793506,0.789610,...,0.773160,0.621645,0.697403,0.812554,0.786147,0.682684,0.692208,0.779221,0.833766,0.813420


In [24]:
chosen_datasets = np.array(chosen_datasets)

In [25]:
ranks = get_ranks_s(chosen_datasets, scores, datasets, return_ranks=True)
ranks_all = get_ranks_s(chosen_datasets, scores, datasets, return_ranks=False)

In [26]:
ranks_aggr = pd.DataFrame(columns=chosen_datasets)

for d in chosen_datasets:
    ranks_aggr[d] = ranks[d].drop(columns=['model']).mean(axis=1)

ranks_aggr.transpose()

,0,1,2,3,4,5,6,7,8,9,...,25,26,27,28,29,30,31,32,33,34
ACSF1,3.100000,21.366667,12.566667,13.700000,1.000000,19.433333,16.433333,3.933333,20.500000,30.600000,...,14.900000,2.000000,8.766667,20.500000,21.733333,6.000000,17.566667,26.300000,15.300000,15.800000
Adiac,2.000000,16.066667,11.700000,15.733333,1.000000,6.666667,29.066667,3.933333,27.966667,22.133333,...,29.033333,3.600000,6.633333,13.466667,18.333333,8.066667,20.933333,31.133333,22.833333,11.733333
ArrowHead,2.666667,21.166667,20.200000,11.800000,5.600000,4.800000,13.066667,19.200000,7.900000,25.433333,...,7.500000,18.433333,5.800000,29.900000,26.300000,7.000000,2.433333,27.333333,24.433333,25.733333
BME,6.266667,22.666667,5.266667,23.266667,16.966667,7.433333,25.766667,15.366667,30.166667,14.733333,...,25.833333,5.700000,20.433333,10.333333,21.400000,12.966667,21.333333,18.566667,10.900000,1.866667
Beef,3.200000,22.766667,11.600000,27.466667,15.766667,3.166667,28.066667,4.466667,28.166667,20.133333,...,27.466667,4.766667,11.933333,14.166667,10.633333,14.900000,20.700000,25.400000,19.800000,7.966667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Wine,13.133333,23.200000,18.900000,16.800000,1.933333,3.166667,16.966667,16.866667,19.100000,20.700000,...,17.633333,14.633333,13.566667,15.833333,19.166667,11.100000,1.266667,30.766667,20.066667,15.200000
WordSynonyms,10.933333,25.466667,11.300000,19.966667,1.933333,1.366667,20.433333,30.833333,8.533333,19.333333,...,14.300000,14.033333,4.700000,25.466667,33.966667,10.466667,3.900000,24.800000,16.100000,13.966667
Worms,3.733333,17.866667,19.100000,13.366667,1.433333,18.966667,24.100000,5.833333,29.166667,21.000000,...,16.700000,1.733333,6.566667,21.900000,27.966667,5.066667,23.533333,16.166667,31.500000,17.066667
WormsTwoClass,4.000000,21.500000,26.500000,25.100000,1.900000,20.566667,24.900000,6.933333,21.566667,20.200000,...,16.266667,2.566667,6.233333,25.900000,19.266667,4.600000,6.100000,17.000000,30.200000,26.866667


In [27]:
ranks_aggr_rounded = ranks_aggr.rank(axis=0)
ranks_aggr_rounded[:5]

,ACSF1,Adiac,ArrowHead,BME,Beef,BeetleFly,BirdChicken,CBF,Car,Chinatown,...,UWaveGestureLibraryAll,UWaveGestureLibraryX,UWaveGestureLibraryY,UWaveGestureLibraryZ,Wafer,Wine,WordSynonyms,Worms,WormsTwoClass,Yoga
0,3.0,2.0,2.0,6.0,3.0,3.0,6.0,24.0,1.0,8.0,...,4.0,2.0,1.0,2.0,2.0,6.0,11.0,3.0,3.0,6.0
1,21.0,16.0,19.0,22.0,22.0,15.0,13.0,32.0,30.0,21.0,...,31.0,29.0,27.0,25.0,14.0,27.0,25.5,16.0,22.0,21.0
2,9.0,11.0,18.0,4.0,9.0,28.0,34.0,34.0,16.0,1.0,...,8.0,5.0,4.0,5.0,15.0,21.0,12.0,18.0,32.0,20.0
3,10.0,15.0,10.0,23.0,31.5,10.0,8.0,17.0,13.0,35.0,...,25.0,24.0,20.0,22.0,19.0,13.0,20.0,10.0,30.0,12.0
4,1.0,1.0,4.0,16.0,14.0,6.0,3.0,2.0,5.0,15.0,...,6.0,3.0,2.0,1.0,1.0,2.0,2.0,1.0,1.0,1.0


In [28]:
def a_optimality_ind(data,
                     sample_size: int,
                     model_list=None,
                     scale_data: bool = True,
                     iter: int = 100,
                     alpha: float = 1e-4,
                     random_state: int = 42,
                     pca_099 = False,
                     **kwargs):

    X = np.asarray(data)
    
    if pca_099:
        X = np.array(StandardScaler().fit_transform(X))
        X = PCA(0.99).fit_transform(X) 

    if scale_data:
        X = StandardScaler().fit_transform(X)

    df = pd.DataFrame(X)
    
    return a_d_optimality_ind(df,
                              sample_size=sample_size,
                              optimality='a',
                              iter=iter,
                              alpha=alpha,
                              random_state=random_state,
                              return_ind=True)


def d_optimality_ind(data,
                     sample_size: int,
                     model_list=None,
                     scale_data: bool = True,
                     iter: int = 100,
                     alpha: float = 1e-4,
                     random_state: int = 42,
                     pca_099 = False,
                     **kwargs):

    X = np.asarray(data)
    
    if pca_099:
        X = np.array(StandardScaler().fit_transform(X))
        X = PCA(0.99).fit_transform(X) 

    if scale_data:
        X = StandardScaler().fit_transform(X)

    df = pd.DataFrame(X)
    return a_d_optimality_ind(df,
                              sample_size=sample_size,
                              optimality='d',
                              iter=iter,
                              alpha=alpha,
                              random_state=random_state,
                              return_ind=True)

Main synthetic Oracle-vs-Broken experiment.

This cell operationalizes the paper's hypothesis:
geometric subset selection only helps when the dataset representation is aligned
with the latent geometry that actually governs model behavior / rank structure.

The synthetic benchmark is built in three steps:
  1. Sample latent dataset vectors Z and latent model vectors P.
  2. Convert their latent distances into smooth fold-level error tables (`scores`).
  3. Construct dataset representations X_repr with either:
       - strong alignment to Z  -> Oracle regime
       - weak/noisy alignment   -> Broken regime

Then we run the exact same `testing_pipeline(...)` as in the real experiments.

In [ ]:
for i in range(100):
    SEED = i
    rng = np.random.default_rng(SEED)

    # match the TSC benchmark scale from the paper as closely as possible.
    NUM_MODELS = 35
    NUM_FOLDS = 30
    

    # synthetic dataset identifiers. We preserve the same dataset-count scale
    # as the main TSC benchmark: N = 112 datasets.
    N = 112
    chosen_datasets = np.array([f"D{i:03d}" for i in range(N)], dtype=str)

    N = len(chosen_datasets)
    all_datasets = chosen_datasets.copy()

    model_names = [f"M{i:02d}" for i in range(NUM_MODELS)]
    fold_cols = [str(i) for i in range(NUM_FOLDS)]

    # latent geometry dimension from the paper's synthetic construction.
    d_latent = 5
    
    # Z: latent dataset positions
    # P: latent model positions
    # the benchmark difficulty / affinity structure is induced entirely by distances
    # between these latent vectors.
    Z = rng.normal(size=(N, d_latent)).astype(np.float32)
    P = rng.normal(size=(NUM_MODELS, d_latent)).astype(np.float32)

    # smooth base error surface:
    # datasets closer to a model in latent space get lower error for that model.
    # shape: (num_datasets, num_models)
    base_err = np.sum((Z[:, None, :] - P[None, :, :]) ** 2, axis=-1)

    
    # add dataset-level and fold-level noise, exactly in the spirit of the paper:
    # - dataset noise perturbs all folds for a dataset,
    # - fold noise adds evaluation variability inside each dataset.
    fold_noise = 0.03
    dataset_noise = 0.02

    # convert latent errors into the same score-table format used by the real codebase:
    # scores[model_name] -> DataFrame with columns ["folds:", "0", ..., "29"].
    scores = {}
    for m, mname in enumerate(model_names):
        E = base_err[:, m][:, None] + dataset_noise * rng.normal(size=(N, 1))
        E = E + fold_noise * rng.normal(size=(N, NUM_FOLDS))
        df = pd.DataFrame(E, columns=fold_cols)
        df.insert(0, "folds:", all_datasets)
        scores[mname] = df

    ranks = get_ranks_s(
        selected_datasets=all_datasets,
        scores=scores,
        all_datasets=all_datasets,
        model_indx=np.arange(NUM_MODELS),
        return_ranks=True,
        fold_num=NUM_FOLDS
    )

    # oracle / aligned representation:
    # random linear projection of the true latent dataset geometry plus tiny noise.
    # distances in this feature space should closely reflect the rank-inducing geometry.
    def make_aligned_features(Z, out_dim, rng, noise=0.01):
        A = rng.normal(size=(Z.shape[1], out_dim)).astype(np.float32) / np.sqrt(Z.shape[1])
        X = (Z @ A) + noise * rng.normal(size=(Z.shape[0], out_dim)).astype(np.float32)
        return X

    def make_broken_features(N, out_dim, rng):
        return rng.normal(size=(N, out_dim)).astype(np.float32)

    
    # weakly aligned ("broken" in the paper's terminology) representation:
    # a small aligned component plus a dominant isotropic noise component.
    # in this regime geometry-based selection should lose much of its advantage.
    def make_weakly_aligned_features(Z, out_dim, rng, signal=0.1, noise=1.0):

        d_latent = Z.shape[1]
        A = rng.normal(size=(d_latent, out_dim)).astype(np.float32) / np.sqrt(d_latent)
        signal_part = (Z @ A).astype(np.float32)

        noise_part = rng.normal(size=(Z.shape[0], out_dim)).astype(np.float32)

        X = signal * signal_part + noise * noise_part
        return X

    
    # switch between the two regimes reported in the paper:
    #   USE_MISALIGNED = False -> Oracle / aligned representation
    #   USE_MISALIGNED = True  -> Broken / weakly aligned representation
    USE_MISALIGNED = True

    D = 64

    X_repr = make_aligned_features(Z, D, rng, noise=0.01) if not USE_MISALIGNED else make_weakly_aligned_features(Z, D, rng,
                                                                                                                  signal=0.15,
                                                                                                                  noise=1.0)

    prep_features = np.zeros((N, 1), dtype=np.float32)

    label_suffix = "ORACLE" if not USE_MISALIGNED else f"BROKEN_F_x{SEED}"

    metods_data_list = [
        [rand_ind_method, prep_features,
        range(2, 21),
        False, False,
        {},
        False, False,
        f'Random_N112_D64_{label_suffix}'],

        [get_more_different_datasets, X_repr,
        range(2, 21),
        False, False,
        {'scale_data': True},
        False, False,
        f'Cosine_FF_N112_D64_{label_suffix}'],

        [k_means_ind, X_repr,
        range(2, 21),
        False, False,
        {'scale_data': True},
        False, False,
        f'KMeans_N112_D64_{label_suffix}'],

        [get_more_different_datasets_euclid, X_repr,
        range(2, 21),
        False, False,
        {'scale_data': True},
        False, False,
        f'Euclid_FF_N112_D64_{label_suffix}'],

        # [a_optimality_ind, X_repr,
        # range(2, 21),
        # False, False,
        # {'scale_data': True, 'iter': 1, 'alpha': 1e-4, 'random_state': 42},
        # False, False,
        # f'A-opt_N112_D64_{label_suffix}'],

        # [d_optimality_ind, X_repr,
        # range(2, 21),
        # False, False,
        # {'scale_data': True, 'iter': 1, 'alpha': 1e-4, 'random_state': 42},
        # False, False,
        # f'D-opt_N112_D64_{label_suffix}'],
    ]

    testing_pipeline(
        chosen_datasets,
        metods_data_list,
        ranks,
        scores,
        all_datasets,
        ratio=0.8,
        models_prefix='',
        load_res=True,
        save_results=True,
        save_checpoints=True,
        test_iter=50
    )
